In [6]:
import json
import re
import time
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

BASE_URL = "https://www.universiteitleiden.nl"
STAFF_URL = f"{BASE_URL}/en/science/environmental-sciences/staff_alphabetically"
OUTPUT_FILE = "leiden_professors.json"
DELAY_SECONDS = 1.0  # be polite and avoid triggering the site's bot protection

# Staff who should be included even though their listed role does not contain "professor"
EXTRA_PROFILES = [
    f"{BASE_URL}/en/staffmembers/glenn-aguilar-hernandez",  # listed as postdoc, but is a professor
    f"{BASE_URL}/en/staffmembers/stefano-merciai",  # listed as researcher, but should be included
    f"{BASE_URL}/en/staffmembers/stefano-cucurachi",  # listed as director of education, but should be included
]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept-Language": "en",
}
SESSION = requests.Session()
SESSION.headers.update(HEADERS)

# Boilerplate paragraphs that are not part of the summary
NOISE = re.compile(r"^(more information about|more research|publications|activities)\b", re.I)


def fetch_page(url):
    """Fetch the HTML content of a page."""
    response = SESSION.get(url, timeout=30)
    response.raise_for_status()
    if "Access blocked" in response.text[:3000]:
        raise RuntimeError(f"Request blocked by the site's protection system: {url}")
    return response.text


def clean(text):
    return re.sub(r"\s+", " ", text).strip()


def extract_professor_links(html):
    """Return profile URLs of staff whose listed role contains 'professor'."""
    soup = BeautifulSoup(html, "html.parser")
    links = []
    for item in soup.select("ul.table-list li"):
        a = item.find("a", href=True)
        if not a:
            continue
        meta = item.find("span", class_="meta")
        role = clean(meta.get_text()) if meta else ""
        if "professor" in role.lower():
            url = urljoin(BASE_URL, a["href"])
            if url not in links:
                links.append(url)
    return links


def extract_profile_details(profile_url, html=None):
    """Extract name, title, keywords, summary, projects, organisation and photo from a profile page."""
    soup = BeautifulSoup(html or fetch_page(profile_url), "html.parser")

    # The page has an empty <h1> in the site header, so read the one inside the profile
    name_tag = soup.select_one("article.profile h1") or soup.select_one("section.basics h1")
    name = clean(name_tag.get_text()) if name_tag else ""

    context = soup.select_one("article.profile p.context")
    title = clean(context.get_text()) if context else ""

    # Keywords: visible tag list under the name, falling back to the page's meta keywords
    keywords = [clean(li.get_text()) for li in soup.select("article.profile ul.tags li") if clean(li.get_text())]
    if not keywords:
        meta_kw = soup.find("meta", attrs={"name": "keywords"})
        if meta_kw and meta_kw.get("content"):
            keywords = [clean(k) for k in meta_kw["content"].split(",") if clean(k)]
    tags = ([title] if title else []) + [k for k in keywords if k != title]

    # Summary: paragraphs of the Overview and Profile tabs, without boilerplate or duplicates
    summary_parts = []
    for section in soup.select("article.profile section.tab"):
        if section.get("data-tab-label") not in ("Overview", "Profile"):
            continue
        for p in section.find_all("p"):
            if p.find_parent(class_="box"):
                continue
            text = clean(p.get_text())
            if text and text != title and not NOISE.match(text) and text not in summary_parts:
                summary_parts.append(text)
    summary = " ".join(summary_parts)

    projects, organisation = [], []
    for box in soup.select("article.profile div.box"):
        heading = box.find("h2")
        box_title = heading.get_text().strip().lower() if heading else ""
        # Projects appear under "Projects" or "Research" depending on the profile
        if "research" in box_title:
            for li in box.select("li"):
                a = li.find("a", href=True)
                if not a:
                    continue
                text = clean((li.find("strong") or li.find("p") or li).get_text())
                url = urljoin(BASE_URL, a["href"])
                if text and all(p["url"] != url for p in projects):
                    projects.append({"title": text, "url": url})
        elif box_title == "organisation":
            organisation = [clean(li.get_text()) for li in box.select("li") if clean(li.get_text())]

    photo_tag = soup.select_one("figure.photo img")
    photo_url = urljoin(BASE_URL, photo_tag["src"]) if photo_tag and photo_tag.get("src") else ""

    return {
        "name": name,
        "title": title,
        "keywords": keywords,
        "tags": tags,
        "organisation": organisation,
        "summary": summary,
        "projects": projects,
        "photo_url": photo_url,
        "profile_url": profile_url,
    }


def main():
    staff_html = fetch_page(STAFF_URL)
    profile_urls = extract_professor_links(staff_html)
    if not profile_urls:
        raise RuntimeError("No professor links found - the page structure may have changed.")
    profile_urls += [u for u in EXTRA_PROFILES if u not in profile_urls]
    print(f"Found {len(profile_urls)} profiles")

    profiles = []
    for url in profile_urls:
        print(f"Processing: {url}")
        try:
            profiles.append(extract_profile_details(url))
        except Exception as exc:
            print(f"  Skipped ({exc})")
        time.sleep(DELAY_SECONDS)

    profiles.sort(key=lambda p: p["name"].split()[-1].lower() if p["name"] else "")
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(profiles, f, indent=2, ensure_ascii=False)
    print(f"Data saved to {OUTPUT_FILE}")
    print(f"Total professors extracted: {len(profiles)}")


main()

Found 50 profiles
Processing: https://www.universiteitleiden.nl/en/staffmembers/valerio-barbarossa
Processing: https://www.universiteitleiden.nl/en/staffmembers/henrik-barmentlo
Processing: https://www.universiteitleiden.nl/en/staffmembers/peter-berrill
Processing: https://www.universiteitleiden.nl/en/staffmembers/koos-biesmeijer
Processing: https://www.universiteitleiden.nl/en/staffmembers/carlos-felipe-blanco-rocha
Processing: https://www.universiteitleiden.nl/en/staffmembers/peter-van-bodegom
Processing: https://www.universiteitleiden.nl/en/staffmembers/jan-boersema
Processing: https://www.universiteitleiden.nl/en/staffmembers/thijs-bosker
Processing: https://www.universiteitleiden.nl/en/staffmembers/marco-cinelli
Processing: https://www.universiteitleiden.nl/en/staffmembers/sebastiaan-deetman
Processing: https://www.universiteitleiden.nl/en/staffmembers/franco-donati
Processing: https://www.universiteitleiden.nl/en/staffmembers/jan-willem-erisman
Processing: https://www.universitei

In [ ]:
import json
from collections import Counter

INPUT_FILE = "leiden_professors.json"
OUTPUT_FILE = "leiden_professors_classified.json"

# Manual corrections for cases the scraped data cannot resolve
RANK_OVERRIDES = {
    "Glenn Aguilar Hernandez": "Assistant professor",
    "Stefano Cucurachi": "Associate professor",
}
FIELD_OVERRIDES = {"Jan Willem Erisman": "environmental_biology"}

# Keyword themes per field (matched on lower-cased tags + summary)
IE_THEMES = {
    "Life cycle assessment": ["life cycle", "life-cycle", "lca", "lcia"],
    "Material flows & critical raw materials": ["material flow", "substance flow", "raw material", "critical metal", "metals", "mining", "stocks"],
    "Circular economy & recycling": ["circular", "recycl", "waste", "reuse", "remanufactur"],
    "Footprints & input-output analysis": ["footprint", "input-output", "input–output", "supply chain", "trade"],
    "Energy & decarbonisation": ["energy", "decarbon", "emission", "climate", "battery", "electric", "hydrogen", "renewable"],
    "Buildings, cities & mobility": ["building", "urban", "city", "cities", "mobility", "transport", "housing"],
    "Consumption & wellbeing": ["consumption", "wellbeing", "well-being", "sufficiency", "lifestyle", "household"],
    "Sustainability transitions & policy": ["policy", "transition", "governance", "economic", "economy", "stakeholder"],
    "Decision analysis & MCDA": ["decision", "multi-criteria", "mcda", "risk assessment", "uncertaint"],
    "Modelling, scenarios & data science": ["modelling", "modeling", "scenario", "software", "machine learning", "integrated assessment", "database", "simulation"],
}
EB_THEMES = {
    "Community ecology & biodiversity": ["biodiversity", "community ecology", "species", "population", "food web", "ecosystem function"],
    "Pollinators & insects": ["pollinat", "insect", "bees", "butterfl", "invertebrate"],
    "Birds, mammals & wildlife": ["bird", "mammal", "wildlife", "herbivor", "predator", "large animal"],
    "Ecotoxicology & chemical risk": ["ecotox", "toxic", "chemical", "contaminant", "nanomaterial", "risk assessment"],
    "Pesticides & plastics pollution": ["pesticide", "plastic", "pollut", "fertili", "nitrogen"],
    "Soil, fungi & microbes": ["soil", "microb", "fung", "decompos", "nutrient", "carbon cycl"],
    "Plant ecology & traits": ["plant", "vegetation", "trait", "grassland", "forest", "tree"],
    "Freshwater & aquatic ecology": ["freshwater", "aquatic", "lake", "river", "water quality", "phytoplankton", "wetland"],
    "Marine ecology": ["marine", "ocean", "coastal", "sea ", "seagrass", "fish"],
    "Conservation & restoration": ["conservation", "restoration", "habitat", "protected area", "extinction", "rewilding"],
    "Ecosystem services & nature-based solutions": ["ecosystem service", "nature-based", "natural capital", "nature’s contributions", "valuation"],
    "Land use & landscape": ["land use", "land-use", "landscape", "agricultur", "farming", "urban"],
    "Genetics, eDNA & evolution": ["dna", "genetic", "genom", "evolution", "metabarcoding", "sequencing"],
    "Spatial & ecological modelling": ["gis", "remote sensing", "spatial", "modelling", "modeling", "simulation", "satellite"],
}
MIN_THEME_HITS = 2
MAX_THEMES = 4


def harmonize_rank(raw_tag, name):
    """Map the free-text role to a small, consistent set of ranks."""
    if name in RANK_OVERRIDES:
        return RANK_OVERRIDES[name]
    t = raw_tag.lower()
    if "professor" not in t:
        return "Researcher"
    if "emeritus" in t:
        return "Professor emeritus"
    if "special appointment" in t:
        return "Professor by special appointment"
    if "assistant" in t:
        return "Assistant professor"
    if "associate" in t:
        return "Associate professor"
    return "Professor"


def theme_hits(text, themes):
    return {theme: sum(text.count(k) for k in kws) for theme, kws in themes.items()}


def departments(profile):
    orgs = " ".join(profile.get("organisation", [])).lower()
    return {
        "industrial_ecology": "industrial ecology" in orgs,
        "environmental_biology": "environmental biology" in orgs,
    }


def classify(profile, text):
    """Department is the primary signal; keyword scores break ties and fill gaps."""
    if profile["name"] in FIELD_OVERRIDES:
        return FIELD_OVERRIDES[profile["name"]], "manual override"
    dept = departments(profile)
    if dept["industrial_ecology"] != dept["environmental_biology"]:
        field = "industrial_ecology" if dept["industrial_ecology"] else "environmental_biology"
        return field, "department"
    ie = sum(theme_hits(text, IE_THEMES).values())
    eb = sum(theme_hits(text, EB_THEMES).values())
    basis = "keywords (both departments listed)" if dept["industrial_ecology"] else "keywords (no department listed)"
    return ("industrial_ecology" if ie >= eb else "environmental_biology"), f"{basis}: IE={ie}, EB={eb}"


with open(INPUT_FILE, encoding="utf-8") as f:
    profiles = json.load(f)

groups = {"industrial_ecology": [], "environmental_biology": []}
for p in profiles:
    raw_tag = p.get("title", "")
    text = f"{' '.join(p['tags'])} {p['summary']}".lower()
    field, basis = classify(p, text)

    themes = EB_THEMES if field == "environmental_biology" else IE_THEMES
    hits = theme_hits(text, themes)
    top = [t for t, n in sorted(hits.items(), key=lambda kv: -kv[1]) if n >= MIN_THEME_HITS][:MAX_THEMES]

    rank = harmonize_rank(raw_tag, p["name"])
    groups[field].append({
        **p,
        "field": field,
        "classification_basis": basis,
        "rank": rank,
        "harmonized_tags": top,
        "original_keywords": p.get("keywords", []),
    })

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(groups, f, indent=2, ensure_ascii=False)

for field, members in groups.items():
    print(f"\n=== {field.replace('_', ' ').title()} ({len(members)}) ===")
    print("Common themes:", dict(Counter(t for m in members for t in m["harmonized_tags"]).most_common()))
    print("Ranks:", dict(Counter(m["rank"] for m in members)))
    for m in members:
        flag = "" if m["classification_basis"] == "department" else f"  [{m['classification_basis']}]"
        print(f"  - {m['name']}: {m['rank']} | {', '.join(m['harmonized_tags'])}{flag}")


=== Industrial Ecology (29) ===
Common themes: {'Sustainable production & consumption': 20, 'Life cycle assessment': 12, 'Energy, climate & mobility': 11, 'Circular economy & material flows': 11, 'Footprints & input-output analysis': 8, 'Decision analysis & modelling': 6}
Ranks: {'Assistant professor': 15, 'Associate professor': 6, 'Professor': 5, 'Professor emeritus': 2, 'Professor by special appointment': 1}
  - Valerio Barbarossa: Assistant professor, Sustainable production & consumption
  - Peter Berrill: Assistant professor, Energy, climate & mobility, Circular economy & material flows, Footprints & input-output analysis
  - Marco Cinelli: Assistant professor, Decision analysis & modelling, Sustainable production & consumption, Life cycle assessment
  - Sónia Martins da Cunha: Assistant professor
  - Sebastiaan Deetman: Assistant professor, Life cycle assessment, Energy, climate & mobility, Sustainable production & consumption
  - Franco Donati: Assistant professor, Decision anal

In [ ]:
import json
from pathlib import Path

# Builds a self-contained dashboard (data embedded) from the classified JSON; open dashboard.html in a browser
data = json.loads(Path("leiden_professors_classified.json").read_text(encoding="utf-8"))
template = Path("dashboard_template.html").read_text(encoding="utf-8")
payload = json.dumps(data, ensure_ascii=False).replace("</", "<\\/")
Path("dashboard.html").write_text(template.replace("__DATA__", payload), encoding="utf-8")
print("Dashboard written to dashboard.html")
